In [ ]:
#STAGE 1

In [ ]:
import pandas as pd
from sklearn.feature_selection import VarianceThreshold

# ==========================================
# LOAD EXPRESSION DATA
# ==========================================

expr = pd.read_csv("/content/FINAL_NSCLC_EXPRESSION.csv")

print("Original Shape:")
print(expr.shape)

# ==========================================
# SAVE MODEL IDS
# ==========================================

model_ids = expr.iloc[:, 0]

# ==========================================
# GENE MATRIX ONLY
# ==========================================

X = expr.iloc[:, 1:]

# ==========================================
# VARIANCE FILTER
# ==========================================

selector = VarianceThreshold(threshold=0.5)

X_filtered = selector.fit_transform(X)

selected_genes = X.columns[selector.get_support()]

# ==========================================
# CREATE FILTERED DATAFRAME
# ==========================================

filtered_expr = pd.DataFrame(
    X_filtered,
    columns=selected_genes
)

filtered_expr.insert(0, "ModelID", model_ids)

print("\nFiltered Shape:")
print(filtered_expr.shape)

print("\nGenes Removed:")
print(X.shape[1] - filtered_expr.shape[1] + 1)

# ==========================================
# SAVE
# ==========================================

filtered_expr.to_csv(
    "/content/NSCLC_expression_filtered.csv",
    index=False
)

print("\nSaved Successfully")

Original Shape:
(140, 19206)

Filtered Shape:
(140, 8147)

Genes Removed:
11059

Saved Successfully


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# ==========================================
# LOAD FILTERED DATA
# ==========================================

expr = pd.read_csv(
    "/content/NSCLC_expression_filtered.csv"
)

print("Original Shape:")
print(expr.shape)

# ==========================================
# SAVE MODEL IDS
# ==========================================

model_ids = expr.iloc[:, 0]

# ==========================================
# GENE MATRIX
# ==========================================

X = expr.iloc[:, 1:]

# ==========================================
# STANDARDIZATION
# ==========================================

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

# ==========================================
# CREATE DATAFRAME
# ==========================================

scaled_expr = pd.DataFrame(
    X_scaled,
    columns=X.columns
)

scaled_expr.insert(
    0,
    "ModelID",
    model_ids
)

print("\nScaled Shape:")
print(scaled_expr.shape)

# ==========================================
# SAVE
# ==========================================

scaled_expr.to_csv(
    "/content/NSCLC_expression_scaled.csv",
    index=False
)

print("\nSaved Successfully!")

Original Shape:
(140, 8147)

Scaled Shape:
(140, 8147)

Saved Successfully!


In [ ]:
#STAGE 2 AutoEncoder

In [ ]:
import pandas as pd
import numpy as np

# ==========================================
# LOAD SCALED EXPRESSION DATA
# ==========================================

expr = pd.read_csv(
    "/content/NSCLC_expression_scaled.csv"
)

print("Dataset Shape:")
print(expr.shape)

# ==========================================
# REMOVE MODEL IDS
# ==========================================

X = expr.iloc[:, 1:].values

print("\nTraining Matrix Shape:")
print(X.shape)

Dataset Shape:
(140, 8147)

Training Matrix Shape:
(140, 8146)


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    BatchNormalization
)
from tensorflow.keras.models import Model

# ==========================================
# INPUT DIMENSION
# ==========================================

input_dim = X.shape[1]

# ==========================================
# ENCODER
# ==========================================

inputs = Input(shape=(input_dim,))

x = Dense(2048, activation='relu')(inputs)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

x = Dense(1024, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

x = Dense(512, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

latent = Dense(
    256,
    activation='relu',
    name="latent_embedding"
)(x)

# ==========================================
# DECODER
# ==========================================

x = Dense(512, activation='relu')(latent)

x = Dense(1024, activation='relu')(x)

x = Dense(2048, activation='relu')(x)

outputs = Dense(
    input_dim,
    activation='linear'
)(x)

# ==========================================
# BUILD MODELS
# ==========================================

autoencoder = Model(inputs, outputs)

encoder = Model(
    inputs,
    latent,
    name="Encoder"
)

# ==========================================
# COMPILE
# ==========================================

autoencoder.compile(
    optimizer='adam',
    loss='mse'
)

# ==========================================
# SUMMARY
# ==========================================

autoencoder.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 8146)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 2048)           │    16,685,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 2048)           │         8,192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 1024)           │     2,098,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 1024)           │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 512)            │       524,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ latent_embedding (Dense)        │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 512)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 1024)           │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 2048)           │     2,099,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 8146)           │    16,691,154 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 38,900,946 (148.40 MB)

 Trainable params: 38,893,778 (148.37 MB)

 Non-trainable params: 7,168 (28.00 KB)

In [ ]:
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint
)

# ==========================================
# CALLBACKS
# ==========================================

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

checkpoint = ModelCheckpoint(
    '/content/best_autoencoder.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

# ==========================================
# TRAIN
# ==========================================

history = autoencoder.fit(
    X,
    X,
    epochs=200,
    batch_size=16,
    validation_split=0.2,
    callbacks=[
        early_stop,
        checkpoint
    ],
    verbose=1
)

Epoch 1/200
6/7 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 1.0651
Epoch 1: val_loss improved from None to 0.94371, saving model to /content/best_autoencoder.keras

Epoch 1: finished saving model to /content/best_autoencoder.keras
7/7 ━━━━━━━━━━━━━━━━━━━━ 14s 1s/step - loss: 1.0173 - val_loss: 0.9437
Epoch 2/200
6/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.9447
Epoch 2: val_loss did not improve from 0.94371
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.9448 - val_loss: 1.7810
Epoch 3/200
5/7 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.8695 
Epoch 3: val_loss did not improve from 0.94371
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 0.8969 - val_loss: 1.6812
Epoch 4/200
6/7 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.8636
Epoch 4: val_loss did not improve from 0.94371
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.8518 - val_loss: 1.7189
Epoch 5/200
6/7 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.8096
Epoch 5: val_loss did not improve from 0.94371
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/st

In [ ]:
encoder.save(
    "/content/pretrained_encoder.keras"
)

print("Encoder saved successfully!")

Encoder saved successfully!


In [ ]:
cancer_embeddings = encoder.predict(X)

print(cancer_embeddings.shape)

5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 66ms/step
(140, 256)


In [ ]:
#Stage 3 — Drug Encoder

In [ ]:
# RDKit
try:
    import rdkit
    print("RDKit:", rdkit.__version__)
except:
    print("RDKit NOT installed")

# Torch
try:
    import torch
    print("Torch:", torch.__version__)
except:
    print("Torch NOT installed")

# PyTorch Geometric
try:
    import torch_geometric
    print("PyG:", torch_geometric.__version__)
except:
    print("PyG NOT installed")

RDKit: 2026.03.2
Torch: 2.11.0+cu128
PyG: 2.7.0


In [ ]:
# ==========================================
# USE NEW PUBCHEM DATASET
# ==========================================

drug_df = full_smiles_df.copy()

print("Drug Dataset Shape:")
print(drug_df.shape)

drug_df.head()

Drug Dataset Shape:
(229, 2)


,Drug,CanonicalSMILES
0,Camptothecin,CCC1(C2=C(COC1=O)C(=O)N3CC4=CC5=CC=CC=C5N=C4C3...
1,Vinblastine,CCC1(CC2CC(C3=C(CCN(C2)C1)C4=CC=CC=C4N3)(C5=C(...
2,Cisplatin,N.N.Cl[Pt]Cl
3,Cytarabine,C1=CN(C(=O)N=C1N)C2C(C(C(O2)CO)O)O
4,Docetaxel,CC1=C2C(C(=O)C3(C(CC4C(C3C(C(C2(C)C)(CC1OC(=O)...


In [ ]:
from rdkit import Chem
from torch_geometric.data import Data
import torch

# ==========================================
# ATOM FEATURES
# ==========================================

def atom_features(atom):
    return [
        atom.GetAtomicNum(),
        atom.GetDegree(),
        atom.GetFormalCharge(),
        atom.GetHybridization().real,
        int(atom.GetIsAromatic())
    ]

# ==========================================
# SMILES → GRAPH
# ==========================================

def smiles_to_graph(smiles):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    x = []

    for atom in mol.GetAtoms():
        x.append(atom_features(atom))

    x = torch.tensor(
        x,
        dtype=torch.float
    )

    edge_index = []

    for bond in mol.GetBonds():

        start = bond.GetBeginAtomIdx()
        end = bond.GetEndAtomIdx()

        edge_index.append([start, end])
        edge_index.append([end, start])

    edge_index = torch.tensor(
        edge_index,
        dtype=torch.long
    ).t().contiguous()

    return Data(
        x=x,
        edge_index=edge_index
    )

# ==========================================
# BUILD ALL DRUG GRAPHS
# ==========================================

drug_graphs = {}

for _, row in drug_df.iterrows():

    drug = row['Drug']
    smiles = row['CanonicalSMILES']

    graph = smiles_to_graph(smiles)

    if graph is not None:
        drug_graphs[drug] = graph

print("Total Graphs Created:")
print(len(drug_graphs))

Total Graphs Created:
229


In [ ]:
from torch_geometric.loader import DataLoader

# ==========================================
# GENERATE EMBEDDINGS
# ==========================================

gnn_model.eval()

drug_embeddings = {}

with torch.no_grad():

    for drug_name, graph in drug_graphs.items():

        loader = DataLoader(
            [graph],
            batch_size=1
        )

        batch = next(iter(loader))

        embedding = gnn_model(batch)

        drug_embeddings[drug_name] = (
            embedding
            .squeeze(0)
            .cpu()
            .numpy()
        )

print("Drug Embeddings Created:")
print(len(drug_embeddings))

Drug Embeddings Created:
229


In [ ]:
import pandas as pd

# ==========================================
# CONVERT TO DATAFRAME
# ==========================================

drug_emb_df = pd.DataFrame.from_dict(
    drug_embeddings,
    orient='index'
)

drug_emb_df.reset_index(
    inplace=True
)

drug_emb_df.rename(
    columns={
        'index':'DRUG_NAME'
    },
    inplace=True
)

print(drug_emb_df.shape)

drug_emb_df.head()

(229, 257)


,DRUG_NAME,0,1,2,3,4,5,6,7,8,...,246,247,248,249,250,251,252,253,254,255
0,Camptothecin,0.117157,0.010270,-0.018386,0.041627,-0.275369,-0.002619,-0.153092,0.261893,-0.072154,...,-0.360099,-0.095934,-0.042255,0.045068,-0.059766,0.221602,-0.020401,0.078823,0.063624,-0.347775
1,Vinblastine,0.108660,0.009380,-0.006567,0.038595,-0.267212,0.002425,-0.156872,0.280283,-0.068922,...,-0.356214,-0.111240,-0.034007,0.065964,-0.055007,0.233747,-0.021007,0.071079,0.060191,-0.349817
2,Cisplatin,0.414457,0.117790,0.527061,0.487769,-0.910482,-0.121388,-0.766233,0.884253,-0.457017,...,-0.780698,-0.485553,-0.402078,0.946866,-0.332825,1.252649,0.061816,-0.057996,0.093703,-1.180949
3,Cytarabine,0.117637,0.008608,0.003159,0.051750,-0.282011,-0.002790,-0.172055,0.288323,-0.066115,...,-0.363420,-0.107619,-0.038944,0.074042,-0.059190,0.249295,-0.018598,0.064286,0.065387,-0.367697
4,Docetaxel,0.110703,0.011319,-0.002308,0.044602,-0.265482,0.001781,-0.162764,0.279339,-0.066512,...,-0.353791,-0.110186,-0.036484,0.073283,-0.053685,0.237441,-0.019278,0.066188,0.063257,-0.351199


In [ ]:
# ==========================================
# MERGE WITH TRAINING DATA
# ==========================================

training_full = training_df.merge(
    drug_emb_df,
    on='DRUG_NAME',
    how='inner'
)

print("Training Shape:")
print(training_full.shape)

Training Shape:
(19207, 530)


In [ ]:
#Stage 4 — GNN Drug Encoder

In [ ]:
import torch
import torch.nn.functional as F

from torch_geometric.nn import (
    GCNConv,
    global_mean_pool
)

# ==========================================
# GNN MODEL
# ==========================================

class DrugGNN(torch.nn.Module):

    def __init__(self):

        super().__init__()

        self.conv1 = GCNConv(5, 64)

        self.conv2 = GCNConv(64, 128)

        self.conv3 = GCNConv(128, 256)

        self.fc = torch.nn.Linear(
            256,
            256
        )

    def forward(self, data):

        x = data.x
        edge_index = data.edge_index
        batch = data.batch

        x = self.conv1(
            x,
            edge_index
        )

        x = F.relu(x)

        x = self.conv2(
            x,
            edge_index
        )

        x = F.relu(x)

        x = self.conv3(
            x,
            edge_index
        )

        x = F.relu(x)

        # Pool all atoms into one vector

        x = global_mean_pool(
            x,
            batch
        )

        x = self.fc(x)

        return x

# ==========================================
# CREATE MODEL
# ==========================================

gnn_model = DrugGNN()

print(gnn_model)

DrugGNN(
  (conv1): GCNConv(5, 64)
  (conv2): GCNConv(64, 128)
  (conv3): GCNConv(128, 256)
  (fc): Linear(in_features=256, out_features=256, bias=True)
)


In [ ]:
from torch_geometric.loader import DataLoader

# ==========================================
# TAKE ONE DRUG
# ==========================================

example_graph = drug_graphs['Erlotinib']

# ==========================================
# CREATE BATCH
# ==========================================

loader = DataLoader(
    [example_graph],
    batch_size=1
)

batch = next(iter(loader))

# ==========================================
# FORWARD PASS
# ==========================================

gnn_model.eval()

with torch.no_grad():

    drug_embedding = gnn_model(batch)

print("Drug Embedding Shape:")
print(drug_embedding.shape)

Drug Embedding Shape:
torch.Size([1, 256])


In [ ]:
cancer_embeddings = encoder.predict(X)

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step


In [ ]:
import pandas as pd

# Load expression file again
expr = pd.read_csv('/content/FINAL_NSCLC_EXPRESSION.csv')

# Get Model IDs
model_ids = expr.iloc[:,0]

# Create embedding dataframe
embedding_df = pd.DataFrame(
    cancer_embeddings
)

embedding_df.insert(
    0,
    'ModelID',
    model_ids
)

print(embedding_df.shape)

embedding_df.head()

(140, 257)


,ModelID,0,1,2,3,4,5,6,7,8,...,246,247,248,249,250,251,252,253,254,255
0,ACH-001113,0.0,0.0,0.000000,0.000000,0.023324,0.204384,0.0,0.000000,0.0,...,0.000000,0.965330,7.802983,0.000000,6.033412,0.0,0.0,1.968587,2.203977,0.000000
1,ACH-000327,0.0,0.0,8.521472,0.000000,2.682854,0.000000,0.0,5.707951,0.0,...,9.984325,0.000000,2.394279,0.000000,0.000000,0.0,0.0,0.000000,0.000000,1.505686
2,ACH-000705,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,...,0.000000,1.135131,6.695018,0.000000,4.344355,0.0,0.0,0.630044,1.345261,0.000000
3,ACH-000769,0.0,0.0,0.000000,2.933872,0.000000,5.080203,0.0,1.007952,0.0,...,0.000000,4.894772,7.563287,3.385449,1.911925,0.0,0.0,0.000000,1.293957,0.000000
4,ACH-000528,0.0,0.0,0.000000,8.689753,0.000000,9.536879,0.0,3.491262,0.0,...,0.000000,0.000000,0.000000,0.919749,5.411538,0.0,0.0,0.449896,0.000000,0.000000


In [ ]:
model = pd.read_csv('/content/Model.csv')

mapping = model[
    ['ModelID', 'SangerModelID']
].dropna()

embedding_df = embedding_df.merge(
    mapping,
    on='ModelID',
    how='inner'
)

print(embedding_df.shape)

embedding_df.head()

(101, 258)


,ModelID,0,1,2,3,4,5,6,7,8,...,247,248,249,250,251,252,253,254,255,SangerModelID
0,ACH-001113,0.0,0.0,0.000000,0.000000,0.023324,0.204384,0.0,0.000000,0.0,...,0.965330,7.802983,0.000000,6.033412,0.0,0.0,1.968587,2.203977,0.000000,SIDM01226
1,ACH-000327,0.0,0.0,8.521472,0.000000,2.682854,0.000000,0.0,5.707951,0.0,...,0.000000,2.394279,0.000000,0.000000,0.0,0.0,0.000000,0.000000,1.505686,SIDM00644
2,ACH-000705,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,...,1.135131,6.695018,0.000000,4.344355,0.0,0.0,0.630044,1.345261,0.000000,SIDM00298
3,ACH-000769,0.0,0.0,0.000000,2.933872,0.000000,5.080203,0.0,1.007952,0.0,...,4.894772,7.563287,3.385449,1.911925,0.0,0.0,0.000000,1.293957,0.000000,SIDM00548
4,ACH-000528,0.0,0.0,0.000000,8.689753,0.000000,9.536879,0.0,3.491262,0.0,...,0.000000,0.000000,0.919749,5.411538,0.0,0.0,0.449896,0.000000,0.000000,SIDM00494


In [ ]:
gdsc = pd.read_csv(
    '/content/FINAL_NSCLC_GDSC.csv'
)

gdsc_ids = set(
    gdsc['SANGER_MODEL_ID']
)

embedding_df = embedding_df[
    embedding_df['SangerModelID']
    .isin(gdsc_ids)
]

print(
    "Final Overlapping Cell Lines:"
)

print(
    embedding_df.shape[0]
)

Final Overlapping Cell Lines:
92


In [ ]:
embedding_df.to_csv(
    '/content/CANCER_EMBEDDINGS.csv',
    index=False
)

print("Saved!")

Saved!


In [ ]:
import pandas as pd

# =====================================
# LOAD
# =====================================

embeddings = pd.read_csv(
    '/content/CANCER_EMBEDDINGS.csv'
)

gdsc = pd.read_csv(
    '/content/FINAL_NSCLC_GDSC.csv'
)

# =====================================
# MERGE
# =====================================

training_df = gdsc.merge(
    embeddings,
    left_on='SANGER_MODEL_ID',
    right_on='SangerModelID',
    how='inner'
)

print("Training Shape:")
print(training_df.shape)

print("\nFirst Columns:")
print(training_df.columns[:20])

Training Shape:
(23281, 274)

First Columns:
Index(['DATASET', 'NLME_RESULT_ID', 'NLME_CURVE_ID', 'CELL_LINE_NAME',
       'SANGER_MODEL_ID', 'CANCER_TYPE', 'DRUG_ID', 'DRUG_NAME',
       'PUTATIVE_TARGET', 'PATHWAY_NAME', 'MIN_CONC', 'MAX_CONC', 'LN_IC50',
       'AUC', 'RMSE', 'Z_SCORE', 'ModelID', '0', '1', '2'],
      dtype='object')


In [ ]:
from torch_geometric.loader import DataLoader

gnn_model.eval()

drug_embeddings = {}

with torch.no_grad():

    for drug_name, graph in drug_graphs.items():

        loader = DataLoader(
            [graph],
            batch_size=1
        )

        batch = next(iter(loader))

        emb = gnn_model(batch)

        drug_embeddings[drug_name] = (
            emb.squeeze(0)
            .numpy()
        )

print(
    "Drug Embeddings Created:"
)

print(
    len(drug_embeddings)
)

Drug Embeddings Created:
298


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import (
    Input,
    Dense,
    MultiHeadAttention,
    LayerNormalization,
    Concatenate,
    GlobalAveragePooling1D
)
from tensorflow.keras.models import Model

In [ ]:
import tensorflow as tf

from tensorflow.keras.layers import (
    Input,
    Dense,
    Reshape,
    MultiHeadAttention,
    LayerNormalization,
    Concatenate,
    Flatten,
    Dropout
)

from tensorflow.keras.models import Model

# ==========================================
# INPUTS
# ==========================================

cancer_input = Input(
    shape=(256,),
    name='CancerEmbedding'
)

drug_input = Input(
    shape=(256,),
    name='DrugEmbedding'
)

# ==========================================
# CONVERT TO SEQUENCES
# ==========================================

cancer_seq = Reshape(
    (1,256)
)(cancer_input)

drug_seq = Reshape(
    (1,256)
)(drug_input)

# ==========================================
# CANCER -> DRUG ATTENTION
# ==========================================

cancer_to_drug = MultiHeadAttention(
    num_heads=4,
    key_dim=64
)(
    query=cancer_seq,
    key=drug_seq,
    value=drug_seq
)

cancer_to_drug = LayerNormalization()(
    cancer_to_drug
)

# ==========================================
# DRUG -> CANCER ATTENTION
# ==========================================

drug_to_cancer = MultiHeadAttention(
    num_heads=4,
    key_dim=64
)(
    query=drug_seq,
    key=cancer_seq,
    value=cancer_seq
)

drug_to_cancer = LayerNormalization()(
    drug_to_cancer
)

# ==========================================
# FLATTEN
# ==========================================

cancer_att = Flatten()(
    cancer_to_drug
)

drug_att = Flatten()(
    drug_to_cancer
)

# ==========================================
# HYBRID FUSION
# ==========================================

fusion = Concatenate()([
    cancer_input,
    drug_input,
    cancer_att,
    drug_att
])

# ==========================================
# PREDICTION HEAD
# ==========================================

x = Dense(
    512,
    activation='relu'
)(fusion)

x = Dropout(0.3)(x)

x = Dense(
    256,
    activation='relu'
)(x)

x = Dropout(0.3)(x)

x = Dense(
    128,
    activation='relu'
)(x)

output = Dense(
    1,
    name='LN_IC50'
)(x)

# ==========================================
# MODEL
# ==========================================

final_model = Model(
    inputs=[
        cancer_input,
        drug_input
    ],
    outputs=output
)

final_model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

final_model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ CancerEmbedding     │ (None, 256)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DrugEmbedding       │ (None, 256)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 1, 256)    │          0 │ DrugEmbedding[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 1, 256)    │          0 │ CancerEmbedding[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 1, 256)    │    263,168 │ reshape_1[0][0],  │
│ (MultiHeadAttentio… │                   │            │ reshape[0][0],    │
│                     │                   │            │ reshape_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 1, 256)    │    263,168 │ reshape[0][0],    │
│ (MultiHeadAttentio… │                   │            │ reshape_1[0][0],  │
│                     │                   │            │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 1, 256)    │        512 │ multi_head_atten… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 1, 256)    │        512 │ multi_head_atten… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 256)       │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 256)       │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 1024)      │          0 │ CancerEmbedding[… │
│ (Concatenate)       │                   │            │ DrugEmbedding[0]… │
│                     │                   │            │ flatten[0][0],    │
│                     │                   │            │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_28 (Dense)    │ (None, 512)       │    524,800 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_14          │ (None, 512)       │          0 │ dense_28[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_29 (Dense)    │ (None, 256)       │    131,328 │ dropout_14[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_15          │ (None, 256)       │          0 │ dense_29[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_30 (Dense)    │ (None, 128)       │     32,896 │ dropout_15[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ LN_IC50 (Dense)     │ (None, 1)         │        129 │ dense_30[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,216,513 (4.64 MB)

 Trainable params: 1,216,513 (4.64 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import pandas as pd

drug_emb_df = pd.DataFrame.from_dict(
    drug_embeddings,
    orient='index'
)

drug_emb_df.reset_index(
    inplace=True
)

drug_emb_df.rename(
    columns={'index':'DRUG_NAME'},
    inplace=True
)

print(drug_emb_df.shape)

drug_emb_df.head()

(298, 257)


,DRUG_NAME,0,1,2,3,4,5,6,7,8,...,246,247,248,249,250,251,252,253,254,255
0,Erlotinib,0.119585,0.009622,-0.007269,0.049428,-0.278787,-0.002709,-0.162820,0.268317,-0.068636,...,-0.355040,-0.095282,-0.041090,0.055840,-0.059056,0.232824,-0.020007,0.070811,0.063998,-0.352596
1,Rapamycin,0.105322,0.008644,0.007907,0.042680,-0.263525,0.006589,-0.165569,0.293029,-0.061097,...,-0.348603,-0.117673,-0.028586,0.086035,-0.051650,0.246719,-0.019368,0.059501,0.060243,-0.353941
2,Sunitinib,0.114235,0.010490,-0.005144,0.045860,-0.270598,0.000207,-0.161276,0.271394,-0.066406,...,-0.351685,-0.102224,-0.038516,0.062586,-0.055393,0.232270,-0.020027,0.069813,0.062923,-0.348987
3,PHA-665752,0.126889,0.015798,0.003152,0.060029,-0.286682,-0.004686,-0.180326,0.281705,-0.078027,...,-0.368186,-0.111884,-0.054012,0.083482,-0.060926,0.258824,-0.017283,0.068594,0.068850,-0.378239
4,MG-132,0.110120,0.012033,0.007134,0.049301,-0.267920,0.003529,-0.168485,0.281064,-0.060809,...,-0.345718,-0.109332,-0.035381,0.079343,-0.051221,0.243617,-0.017377,0.059828,0.060782,-0.351502


In [ ]:
training_full = training_df.merge(
    drug_emb_df,
    on='DRUG_NAME',
    how='inner'
)

print("Training Shape:")
print(training_full.shape)

Training Shape:
(19207, 530)


In [ ]:
import numpy as np

# ======================================
# CANCER FEATURES
# ======================================

cancer_cols = [str(i) for i in range(256)]

X_cancer = training_full[
    cancer_cols
].values

# ======================================
# DRUG FEATURES
# ======================================

drug_cols = [col for col in training_full.columns
             if isinstance(col, int)]

X_drug = training_full[
    drug_cols
].values

# ======================================
# TARGET
# ======================================

y = training_full[
    'LN_IC50'
].values

print("Cancer Shape:")
print(X_cancer.shape)

print("\nDrug Shape:")
print(X_drug.shape)

print("\nTarget Shape:")
print(y.shape)

Cancer Shape:
(19207, 256)

Drug Shape:
(19207, 256)

Target Shape:
(19207,)


In [ ]:
from sklearn.model_selection import train_test_split

Xc_train, Xc_test, Xd_train, Xd_test, y_train, y_test = train_test_split(
    X_cancer,
    X_drug,
    y,
    test_size=0.2,
    random_state=42
)

print(Xc_train.shape)
print(Xc_test.shape)

(15365, 256)
(3842, 256)


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

history = final_model.fit(

    [Xc_train, Xd_train],

    y_train,

    validation_split=0.2,

    epochs=100,

    batch_size=128,

    callbacks=[early_stop],

    verbose=1
)

Epoch 1/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 10s 51ms/step - loss: 5.5366 - mae: 1.8210 - val_loss: 8.6523 - val_mae: 2.4582
Epoch 2/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 6.2918 - mae: 1.9540 - val_loss: 6.0120 - val_mae: 1.9379
Epoch 3/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 5.6619 - mae: 1.8539 - val_loss: 7.6819 - val_mae: 2.3232
Epoch 4/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 5.4890 - mae: 1.8300 - val_loss: 6.0798 - val_mae: 2.0266
Epoch 5/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 5.3810 - mae: 1.8118 - val_loss: 6.1661 - val_mae: 2.0563
Epoch 6/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 5.2489 - mae: 1.7945 - val_loss: 6.3565 - val_mae: 2.1037
Epoch 7/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 5.0017 - mae: 1.7444 - val_loss: 5.5768 - val_mae: 1.8954
Epoch 8/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 4.9812 - mae: 1.7433 - val_loss: 5.5185 - val_mae: 1.9249
Epoch 9/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 4.9

In [ ]:
predictions = final_model.predict(
    [Xc_test, Xd_test]
)

121/121 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step


In [ ]:
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from scipy.stats import pearsonr
import numpy as np

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        predictions
    )
)

mae = mean_absolute_error(
    y_test,
    predictions
)

r2 = r2_score(
    y_test,
    predictions
)

pearson_corr, _ = pearsonr(
    y_test,
    predictions.flatten()
)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)
print("Pearson:", pearson_corr)

RMSE: 2.2032191517039834
MAE: 1.7915458446847525
R2: 0.3755846388286401
Pearson: 0.6716645725277183


In [ ]:
final_model.save(
    "/content/Final_NSCLC_Drug_Response_Model.keras"
)

In [ ]:
# ==========================================
# CANCER EMBEDDINGS
# ==========================================

cancer_cols = [str(i) for i in range(256)]

X_cancer = training_full[
    cancer_cols
].values

# ==========================================
# DRUG EMBEDDINGS
# ==========================================

drug_cols = list(range(256))

X_drug = training_full[
    drug_cols
].values

# ==========================================
# TARGET
# ==========================================

y = training_full[
    'LN_IC50'
].values

print("Cancer Shape:")
print(X_cancer.shape)

print("\nDrug Shape:")
print(X_drug.shape)

print("\nTarget Shape:")
print(y.shape)

Cancer Shape:
(19207, 256)

Drug Shape:
(19207, 256)

Target Shape:
(19207,)


In [ ]:
#For end to end Training:

In [ ]:
# ==========================================
# MERGE RAW GENE EXPRESSION
# ==========================================

expr_full = scaled_expr.copy()

end2end_df = training_df[
    ['ModelID', 'DRUG_NAME', 'LN_IC50']
].merge(
    expr_full,
    on='ModelID',
    how='inner'
)

print(end2end_df.shape)

end2end_df.head()

(23281, 8149)


,ModelID,DRUG_NAME,LN_IC50,TSPAN6 (7105),CFH (3075),FUCA2 (2519),GCLC (2729),NIPAL3 (57185),LAS1L (81887),ENPP4 (22875),...,ARHGAP11B (89839),NOTCH2NLB (100996763),ASDURF (110599588),NOTCH2NLR (101929796),DERPC (113455421),NOTCH2NLC (100996717),H3C2 (8358),NPBWR1 (2831),F8A2 (474383),F8A1 (8263)
0,ACH-000867,Camptothecin,0.528845,-1.848165,-0.931338,-1.220284,-0.104286,0.180770,-0.285528,0.500907,...,0.039229,-1.171327,-0.975860,-1.464220,-0.174309,-1.090399,0.459767,-0.410063,0.371784,-1.137942
1,ACH-000392,Camptothecin,-1.629546,0.864429,0.303041,-0.736704,-0.742150,-1.032972,-0.908757,0.736770,...,-1.452743,-0.965965,-0.706943,-0.191827,-0.008180,-1.278477,-0.773044,0.084037,-0.201201,-1.518028
2,ACH-000662,Camptothecin,-1.762080,1.001316,-1.216566,0.058513,-0.991798,0.359866,1.211372,0.170742,...,-0.591483,0.053344,-1.185416,0.289543,-1.036150,-0.589317,-0.119847,-0.607097,-0.757256,0.085598
3,ACH-000769,Camptothecin,-1.157607,0.720820,-1.159283,0.004505,0.016968,-0.310588,-0.232841,0.173370,...,0.274256,2.037860,-0.563779,3.022382,0.277748,1.366609,-0.398641,0.138544,0.101572,-0.386293
4,ACH-000589,Camptothecin,-1.839524,0.347131,-1.148448,0.410503,-0.805731,-0.650976,0.367125,-1.503398,...,-0.241024,1.674314,-1.527457,-0.005556,-0.910189,1.769442,1.437822,-0.559463,0.499740,-2.328072


In [ ]:
# ==========================================
# ADD DRUG EMBEDDINGS
# ==========================================

end2end_df = end2end_df.merge(
    drug_emb_df,
    on='DRUG_NAME',
    how='inner'
)

print(end2end_df.shape)

(19207, 8405)


In [ ]:
# ==========================================
# GENE FEATURES
# ==========================================

gene_cols = scaled_expr.columns[1:]

X_genes = end2end_df[
    gene_cols
].values.astype('float32')

# ==========================================
# DRUG FEATURES
# ==========================================

drug_cols = list(range(256))

X_drug = end2end_df[
    drug_cols
].values.astype('float32')

# ==========================================
# TARGET
# ==========================================

y = end2end_df[
    'LN_IC50'
].values.astype('float32')

print("Genes:", X_genes.shape)
print("Drug :", X_drug.shape)
print("Target:", y.shape)

Genes: (19207, 8146)
Drug : (19207, 256)
Target: (19207,)


In [ ]:
from sklearn.model_selection import train_test_split

Xg_train, Xg_test, Xd_train, Xd_test, y_train, y_test = train_test_split(
    X_genes,
    X_drug,
    y,
    test_size=0.2,
    random_state=42
)

print(Xg_train.shape)
print(Xg_test.shape)

(15365, 8146)
(3842, 8146)


In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model

# ==========================================
# LOAD PRETRAINED ENCODER
# ==========================================

encoder = load_model(
    "/content/pretrained_encoder.keras"
)

# IMPORTANT
encoder.trainable = True

# ==========================================
# INPUTS
# ==========================================

gene_input = Input(
    shape=(8146,),
    name="GeneInput"
)

drug_input = Input(
    shape=(256,),
    name="DrugInput"
)

# ==========================================
# ENCODER
# ==========================================

gene_embedding = encoder(
    gene_input
)

# ==========================================
# ATTENTION
# ==========================================

gene_seq = Reshape((1,256))(gene_embedding)
drug_seq = Reshape((1,256))(drug_input)

gene_to_drug = MultiHeadAttention(
    num_heads=4,
    key_dim=64
)(
    gene_seq,
    drug_seq
)

drug_to_gene = MultiHeadAttention(
    num_heads=4,
    key_dim=64
)(
    drug_seq,
    gene_seq
)

gene_att = Flatten()(gene_to_drug)
drug_att = Flatten()(drug_to_gene)

fusion = Concatenate()([
    gene_embedding,
    drug_input,
    gene_att,
    drug_att
])

# ==========================================
# REGRESSOR
# ==========================================

x = Dense(
    512,
    activation='relu'
)(fusion)

x = Dropout(0.3)(x)

x = Dense(
    256,
    activation='relu'
)(x)

x = Dropout(0.3)(x)

x = Dense(
    128,
    activation='relu'
)(x)

output = Dense(
    1,
    name='LN_IC50'
)(x)

end2end_model = Model(
    [gene_input, drug_input],
    output
)

end2end_model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

end2end_model.summary()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ GeneInput           │ (None, 8146)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Encoder             │ (None, 256)       │ 19,453,696 │ GeneInput[0][0]   │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DrugInput           │ (None, 256)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_2 (Reshape) │ (None, 1, 256)    │          0 │ Encoder[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_3 (Reshape) │ (None, 1, 256)    │          0 │ DrugInput[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 1, 256)    │    263,168 │ reshape_2[0][0],  │
│ (MultiHeadAttentio… │                   │            │ reshape_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 1, 256)    │    263,168 │ reshape_3[0][0],  │
│ (MultiHeadAttentio… │                   │            │ reshape_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 256)       │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_3 (Flatten) │ (None, 256)       │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 1024)      │          0 │ Encoder[0][0],    │
│ (Concatenate)       │                   │            │ DrugInput[0][0],  │
│                     │                   │            │ flatten_2[0][0],  │
│                     │                   │            │ flatten_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_31 (Dense)    │ (None, 512)       │    524,800 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_18          │ (None, 512)       │          0 │ dense_31[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_32 (Dense)    │ (None, 256)       │    131,328 │ dropout_18[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_19          │ (None, 256)       │          0 │ dense_32[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_33 (Dense)    │ (None, 128)       │     32,896 │ dropout_19[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ LN_IC50 (Dense)     │ (None, 1)         │        129 │ dense_33[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 20,669,185 (78.85 MB)

 Trainable params: 20,662,017 (78.82 MB)

 Non-trainable params: 7,168 (28.00 KB)

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# ==========================================
# COMPILE MODEL
# ==========================================

end2end_model.compile(
    optimizer=Adam(
        learning_rate=1e-4
    ),
    loss='mse',
    metrics=['mae']
)

# ==========================================
# EARLY STOPPING
# ==========================================

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

# ==========================================
# TRAIN
# ==========================================

history = end2end_model.fit(

    [Xg_train, Xd_train],

    y_train,

    validation_split=0.2,

    epochs=100,

    batch_size=64,

    callbacks=[early_stop],

    verbose=1
)

Epoch 1/100
193/193 ━━━━━━━━━━━━━━━━━━━━ 21s 49ms/step - loss: 7.8140 - mae: 2.1598 - val_loss: 7.5287 - val_mae: 2.1387
Epoch 2/100
193/193 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 7.2917 - mae: 2.0733 - val_loss: 7.5192 - val_mae: 2.1599
Epoch 3/100
193/193 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 7.2094 - mae: 2.0622 - val_loss: 7.7991 - val_mae: 2.2370
Epoch 4/100
193/193 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 7.2084 - mae: 2.0636 - val_loss: 7.7625 - val_mae: 2.2382
Epoch 5/100
193/193 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 7.0853 - mae: 2.0468 - val_loss: 7.8444 - val_mae: 2.2463
Epoch 6/100
193/193 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 7.0992 - mae: 2.0472 - val_loss: 7.4645 - val_mae: 2.1552
Epoch 7/100
193/193 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 7.0660 - mae: 2.0384 - val_loss: 7.5199 - val_mae: 2.1767
Epoch 8/100
193/193 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 6.9907 - mae: 2.0308 - val_loss: 7.8587 - val_mae: 2.2762
Epoch 9/100
193/193 ━━━━━━━━━━━━━━━━━━━━ 2

In [ ]:
predictions = end2end_model.predict(
    [Xg_test, Xd_test]
)

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from scipy.stats import pearsonr
import numpy as np

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        predictions
    )
)

mae = mean_absolute_error(
    y_test,
    predictions
)

r2 = r2_score(
    y_test,
    predictions
)

pearson_corr, _ = pearsonr(
    y_test,
    predictions.flatten()
)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)
print("Pearson:", pearson_corr)

121/121 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step
RMSE: 2.5346851351969977
MAE: 2.0047736167907715
R2: 0.17356973886489868
Pearson: 0.4396729


In [ ]:
#Failed in end to end

In [ ]:
#increasing the atomic variables:

In [ ]:
def atom_features(atom):

    return [

        atom.GetAtomicNum(),              # Atomic Number

        atom.GetDegree(),                 # Degree

        atom.GetFormalCharge(),           # Formal Charge

        atom.GetHybridization().real,     # Hybridization

        int(atom.GetIsAromatic()),        # Aromatic

        atom.GetTotalNumHs(),             # Hydrogens

        atom.GetImplicitValence(),        # Valence

        int(atom.IsInRing()),             # Ring Membership

        atom.GetMass() / 100              # Atomic Mass
    ]

In [ ]:
drug_graphs = {}

for _, row in drug_df.iterrows():

    drug = row['Drug']
    smiles = row['CanonicalSMILES']

    graph = smiles_to_graph(smiles)

    if graph is not None:
        drug_graphs[drug] = graph

print(len(drug_graphs))

229


[07:28:58] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[07:28:58] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[07:28:58] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[07:28:58] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[07:28:58] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[07:28:58] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[07:28:58] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[07:28:58] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[07:28:58] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[07:28:58] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[07:28:58] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[07:28:58] DEPRECATIO

In [ ]:
class DrugGNN(torch.nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = GCNConv(9,64)

        self.conv2 = GCNConv(64,128)

        self.conv3 = GCNConv(128,256)

        self.fc = torch.nn.Linear(256,256)

In [ ]:
sample_graph = next(iter(drug_graphs.values()))
print(sample_graph.x.shape)

torch.Size([26, 9])


In [ ]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [ ]:
print(DrugGNN)

<class '__main__.DrugGNN'>


In [ ]:
import torch
import torch.nn.functional as F

from torch_geometric.nn import (
    GCNConv,
    global_mean_pool
)

class DrugGNN(torch.nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = GCNConv(9, 64)

        self.conv2 = GCNConv(64, 128)

        self.conv3 = GCNConv(128, 256)

        self.fc = torch.nn.Linear(
            256,
            256
        )

    def forward(self, data):

        x = data.x
        edge_index = data.edge_index

        x = self.conv1(
            x,
            edge_index
        )

        x = F.relu(x)

        x = self.conv2(
            x,
            edge_index
        )

        x = F.relu(x)

        x = self.conv3(
            x,
            edge_index
        )

        x = F.relu(x)

        # single graph pooling
        x = x.mean(
            dim=0,
            keepdim=True
        )

        x = self.fc(x)

        return x

In [ ]:
drug_gnn = DrugGNN().to(device)

sample_drug = next(iter(drug_graphs.values()))

sample_drug = sample_drug.to(device)

with torch.no_grad():
    emb = drug_gnn(sample_drug)

print(emb.shape)

torch.Size([1, 256])


In [ ]:
print(emb.shape)

torch.Size([1, 256])


In [ ]:
drug_gnn = DrugGNN().to(device)

drug_embeddings = {}

for drug, graph in drug_graphs.items():

    graph = graph.to(device)

    with torch.no_grad():
        emb = drug_gnn(graph)

    drug_embeddings[drug] = emb.cpu().numpy().flatten()

print("Drug Embeddings Created:")
print(len(drug_embeddings))

Drug Embeddings Created:
229


In [ ]:
drug_emb_df = pd.DataFrame.from_dict(
    drug_embeddings,
    orient='index'
)

drug_emb_df.reset_index(inplace=True)

drug_emb_df.rename(
    columns={'index': 'DRUG_NAME'},
    inplace=True
)

print(drug_emb_df.shape)

drug_emb_df.head()

(229, 257)


,DRUG_NAME,0,1,2,3,4,5,6,7,8,...,246,247,248,249,250,251,252,253,254,255
0,Camptothecin,0.066138,0.086075,-0.112495,-0.065638,0.084328,-0.125935,0.268776,0.104261,-0.177487,...,-0.577893,0.002754,-0.232229,-0.187751,0.472388,-0.134672,0.221360,0.168666,-0.092113,-0.308836
1,Vinblastine,0.076164,0.079922,-0.112459,-0.050765,0.074819,-0.103896,0.267558,0.112403,-0.201057,...,-0.628522,0.015468,-0.230906,-0.182355,0.464149,-0.128730,0.230116,0.191654,-0.087362,-0.307260
2,Cisplatin,-0.157195,0.571545,-0.601366,-0.868561,0.489298,-0.905885,0.460815,-0.167191,-0.684068,...,-1.608927,-0.027976,-0.587100,-0.854369,1.295639,-0.437172,0.244484,0.952557,-0.594505,-0.815453
3,Cytarabine,0.080681,0.076485,-0.116597,-0.052117,0.083180,-0.122120,0.271298,0.108353,-0.202722,...,-0.619184,0.014318,-0.240643,-0.194993,0.481715,-0.134128,0.230438,0.200062,-0.092238,-0.313114
4,Docetaxel,0.077042,0.080769,-0.116669,-0.053903,0.080182,-0.111062,0.263628,0.105931,-0.192447,...,-0.618260,0.017214,-0.229147,-0.184873,0.458699,-0.128598,0.225741,0.190163,-0.091822,-0.302739


In [ ]:
training_full = training_df.merge(
    drug_emb_df,
    on='DRUG_NAME',
    how='inner'
)

print(training_full.shape)

(19207, 530)


In [ ]:
# ==========================================
# CANCER EMBEDDINGS
# ==========================================

cancer_cols = [str(i) for i in range(256)]

X_cancer = training_full[
    cancer_cols
].values.astype('float32')

# ==========================================
# DRUG EMBEDDINGS
# ==========================================

drug_cols = list(range(256))

X_drug = training_full[
    drug_cols
].values.astype('float32')

# ==========================================
# TARGET
# ==========================================

y = training_full[
    'LN_IC50'
].values.astype('float32')

print(X_cancer.shape)
print(X_drug.shape)
print(y.shape)

(19207, 256)
(19207, 256)
(19207,)


In [ ]:
from sklearn.model_selection import train_test_split

Xc_train, Xc_test, Xd_train, Xd_test, y_train, y_test = train_test_split(
    X_cancer,
    X_drug,
    y,
    test_size=0.2,
    random_state=42
)

print(Xc_train.shape)
print(Xc_test.shape)

(15365, 256)
(3842, 256)


In [ ]:
history = final_model.fit(
    [Xc_train, Xd_train],
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 7.5755 - mae: 2.1261 - val_loss: 7.7869 - val_mae: 2.2352
Epoch 2/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 6.9874 - mae: 2.0196 - val_loss: 7.7001 - val_mae: 2.2186
Epoch 3/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 6.9837 - mae: 2.0230 - val_loss: 7.5827 - val_mae: 2.1843
Epoch 4/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 6.9610 - mae: 2.0141 - val_loss: 7.8438 - val_mae: 2.2592
Epoch 5/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 6.9290 - mae: 2.0196 - val_loss: 7.8367 - val_mae: 2.2582
Epoch 6/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 6.8980 - mae: 2.0093 - val_loss: 8.0237 - val_mae: 2.3038
Epoch 7/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 6.9011 - mae: 2.0143 - val_loss: 7.3392 - val_mae: 2.1182
Epoch 8/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 6.9126 - mae: 2.0153 - val_loss: 7.5946 - val_mae: 2.2062
Epoch 9/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 6.824

In [ ]:
predictions = final_model.predict(
    [Xc_test, Xd_test]
)

121/121 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [ ]:
predictions = final_model.predict(
    [Xc_test, Xd_test]
)

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from scipy.stats import pearsonr
import numpy as np

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        predictions
    )
)

mae = mean_absolute_error(
    y_test,
    predictions
)

r2 = r2_score(
    y_test,
    predictions
)

pearson_corr, _ = pearsonr(
    y_test,
    predictions.flatten()
)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)
print("Pearson:", pearson_corr)

121/121 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
RMSE: 2.7537253029538764
MAE: 2.2162435054779053
R2: 0.024562597274780273
Pearson: 0.28836617
